# First PyCUDA example: using a CUDA kernel

This is the first notebook in the five-example PyCUDA sequence. It follows the explicit `SourceModule` approach of the original vector-addition notebook and reuses **the exact `vector_add` kernel from Chapter 2**.

We compute $c_i=a_i+b_i$ with Python managing the host program: create NumPy arrays, allocate device buffers, copy the inputs, compile and launch the CUDA kernel, retrieve the result, and validate it. The default inputs and vector length also match the CUDA C++ example.

Run the cells from top to bottom. Results and timings are displayed directly in the notebook; no figures, spreadsheets, or result files are exported.


## 1. Prepare the GPU runtime

In Google Colab, select **Runtime → Change runtime type → GPU** and connect. The following commands inspect the assigned GPU and CUDA compiler on the remote machine. `SourceModule` needs `nvcc`; a GPU runtime alone does not establish that the compiler is available.


In [1]:
!nvidia-smi
!nvcc --version


Wed Oct 14 05:26:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Install PyCUDA if necessary. The installation may build the package and take several minutes. Continue only after installation succeeds.


In [2]:
!pip install pycuda


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 67.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 12.5 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp313-cp313-linux_x86_64.whl size=5281388 sha256=8228d7bf839566460fa235138ccfec5c7bf5ae8bdb264a1733cd957827327269
  Stored in directory: /root/.cache/pip/wheels/ce/26/46/c519675fcb0e5e17bab8e85b6676528c40d12d794182340e85
Successfully built pycuda


## 2. Initialize PyCUDA

`pycuda.autoinit` creates the CUDA context used by this standalone example. `cuda` provides the driver operations; `SourceModule` compiles the kernel source into a module. CUDA errors are reported as Python exceptions.


In [3]:
import sys
import numpy as np
import pycuda
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

device = cuda.Context.get_device()
print('GPU:', device.name())
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('PyCUDA:', pycuda.VERSION_TEXT)


GPU: Tesla T4
Python: 3.13.15
NumPy: 2.1.3
PyCUDA: 2026.1


## 3. Choose the vector length and launch size

Use $N=100003$ and 256 threads per block, as in Chapter 2. The length is deliberately not divisible by the block size. `iDivUp(N, BLOCKSIZE)` computes the number of blocks by rounding the division upward.

The kernel uses 32-bit integer indexing. The checks below restrict the configuration to valid device limits and keep the padded launch indices within that range.


In [4]:
N = 100003
BLOCKSIZE = 256

def iDivUp(a, b):
    return (a + b - 1) // b

assert 1 <= BLOCKSIZE <= min(
    device.get_attribute(cuda.device_attribute.MAX_THREADS_PER_BLOCK),
    device.get_attribute(cuda.device_attribute.MAX_BLOCK_DIM_X),
)
assert 1 <= N <= np.iinfo(np.int32).max - (BLOCKSIZE - 1)

number_of_blocks = iDivUp(N, BLOCKSIZE)
assert number_of_blocks <= device.get_attribute(
    cuda.device_attribute.MAX_GRID_DIM_X
)
block = (BLOCKSIZE, 1, 1)
grid = (number_of_blocks, 1, 1)

print(f'Elements: {N}')
print(f'Blocks: {number_of_blocks}; threads per block: {BLOCKSIZE}')
print(f'Threads with no output element: {number_of_blocks * BLOCKSIZE - N}')


Elements: 100003
Blocks: 391; threads per block: 256
Threads with no output element: 93


## 4. Prepare the host arrays

The inputs reproduce the Chapter 2 initialization: $a_i=0.25(i\bmod101)$ and $b_i=0.50(i\bmod37)$. Both are contiguous, one-dimensional `float32` arrays, matching CUDA's `float` pointers. `h_c` will receive the GPU result; `np.empty_like` allocates its storage without initializing the entries.


In [5]:
indices = np.arange(N, dtype=np.int32)
h_a = np.float32(0.25) * (indices % 101).astype(np.float32)
h_b = np.float32(0.50) * (indices % 37).astype(np.float32)
h_c = np.empty_like(h_a)

assert h_a.dtype == h_b.dtype == h_c.dtype == np.float32
assert h_a.flags.c_contiguous and h_b.flags.c_contiguous
print(f'Bytes per vector: {h_a.nbytes}')


Bytes per vector: 400012


## 5. Allocate device memory and transfer the inputs

Each `cuda.mem_alloc` call allocates the requested number of **bytes** and returns a `DeviceAllocation` object that can be passed to the kernel. Allocation and copying are separate operations. The first argument of `cuda.memcpy_htod` is the destination device allocation; the second is the source host array.

No input copy is needed for `d_c`: the kernel will write every valid output element.


In [6]:
d_a = cuda.mem_alloc(h_a.nbytes)
d_b = cuda.mem_alloc(h_b.nbytes)
d_c = cuda.mem_alloc(h_c.nbytes)

cuda.memcpy_htod(d_a, h_a)
cuda.memcpy_htod(d_b, h_b)


## 6. Compile the unchanged CUDA kernel

The source below is identical to the kernel in the Chapter 2 notebook, including argument order `(a, b, c, n)`, thread indexing, bounds check, and `extern "C"` declaration. It is embedded in a Python string so that this notebook needs no external header file.

`SourceModule` invokes `nvcc` when compilation is needed and loads the compiled module. Because the source already declares `extern "C"`, `no_extern_c=True` prevents PyCUDA from adding another wrapper. `get_function('vector_add')` retrieves a Python-callable reference to the compiled kernel. Compilation is performed outside the timing interval below.


In [7]:
kernel_source = r"""
extern "C" __global__
void vector_add(const float *a, const float *b, float *c, int n)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}
"""

module = SourceModule(
    kernel_source, options=["-std=c++14"], no_extern_c=True
)
vector_add = module.get_function("vector_add")


## 7. Launch the kernel

The positional arguments follow the CUDA declaration. The `block` and `grid` keywords replace the launch configuration supplied through `<<<...>>>` in CUDA C++.

Python's `int` does not specify the fixed storage type required by this kernel argument. `np.int32(N)` explicitly supplies a 32-bit value matching CUDA's `int`.

The launch is asynchronous with respect to the host. The explicit `cuda.Context.synchronize()` waits for submitted work in this context and allows execution errors to be reported before the result is retrieved. It mirrors the explicit wait in the first CUDA C++ example; the following blocking copy would also wait for the required preceding work.


In [8]:
vector_add(d_a, d_b, d_c, np.int32(N), block=block, grid=grid)
cuda.Context.synchronize()


## 8. Retrieve and validate every output

`cuda.memcpy_dtoh` takes the destination host array first and the source device allocation second. The NumPy reference is computed independently on the CPU. Validation checks all entries, including those in the partial final block, and rejects nonfinite output values.


In [9]:
cuda.memcpy_dtoh(h_c, d_c)
h_reference = h_a + h_b

if not np.all(np.isfinite(h_c)):
    raise AssertionError('The GPU result contains nonfinite values.')
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)
max_abs_error = float(np.max(np.abs(h_c - h_reference)))
print(f'Validation passed for all {N} elements.')
print(f'Maximum absolute error: {max_abs_error:.3e}')


Validation passed for all 100003 elements.
Maximum absolute error: 0.000e+00


## 9. Display the computed sum

The `c[i] (GPU)` column comes from `h_c`, the result copied from device memory. Display the first 12 and last 3 entries so that the output includes the final, partially filled block. The NumPy column is shown separately for comparison.


In [10]:
shown_indices = sorted(set(range(min(12, N))) | set(range(max(0, N - 3), N)))
print(f"{'i':>8} {'a[i]':>12} {'b[i]':>12} {'c[i] (GPU)':>15} {'NumPy':>12}")
for i in shown_indices:
    print(f'{i:8d} {h_a[i]:12.6f} {h_b[i]:12.6f} '
          f'{h_c[i]:15.6f} {h_reference[i]:12.6f}')


       i         a[i]         b[i]      c[i] (GPU)        NumPy
       0     0.000000     0.000000        0.000000     0.000000
       1     0.250000     0.500000        0.750000     0.750000
       2     0.500000     1.000000        1.500000     1.500000
       3     0.750000     1.500000        2.250000     2.250000
       4     1.000000     2.000000        3.000000     3.000000
       5     1.250000     2.500000        3.750000     3.750000
       6     1.500000     3.000000        4.500000     4.500000
       7     1.750000     3.500000        5.250000     5.250000
       8     2.000000     4.000000        6.000000     6.000000
       9     2.250000     4.500000        6.750000     6.750000
      10     2.500000     5.000000        7.500000     7.500000
      11     2.750000     5.500000        8.250000     8.250000
  100000     2.500000    13.000000       15.500000    15.500000
  100001     2.750000    13.500000       16.250000    16.250000
  100002     3.000000    14.000000      

## 10. Optional: measure the device interval

The previous cells already complete and validate the example. This optional measurement follows the CUDA-event approach in the original PyCUDA notebook. Warm-up launches precede timing; a batch of repeated launches gives an average device interval per launch.

`stop.synchronize()` waits until the stream reaches the stop event. This makes the two event timestamps available; it is not a requirement to finish unrelated work in other streams. `start.time_till(stop)` returns milliseconds. Compilation, allocation, transfers, validation, and printing lie outside the interval.

For this short kernel, the interval may include GPU idle gaps between Python submissions. It is not a CPU--GPU speedup or the end-to-end time of vector addition. Repeated launches use the same arrays, so cache reuse may also influence the measurement.


In [11]:
REPEATS = 100
assert REPEATS > 0

for _ in range(3):
    vector_add(d_a, d_b, d_c, np.int32(N), block=block, grid=grid)
cuda.Context.synchronize()

start = cuda.Event()
stop = cuda.Event()
start.record()
for _ in range(REPEATS):
    vector_add(d_a, d_b, d_c, np.int32(N), block=block, grid=grid)
stop.record()
stop.synchronize()

elapsed_ms = start.time_till(stop)
mean_ms = elapsed_ms / REPEATS
print(f'Total device interval for {REPEATS} launches: {elapsed_ms:.6f} ms')
print(f'Mean device interval per launch: {mean_ms:.6f} ms')


Total device interval for 100 launches: 1.082432 ms
Mean device interval per launch: 0.010824 ms


## 11. Release the device buffers

Run this cell when finished with the example. The earlier synchronization has completed the GPU work. After freeing these allocations, rerun the notebook from the parameter cell onward before launching the kernel again. `pycuda.autoinit` handles context cleanup when the Python process exits.


In [12]:
d_a.free()
d_b.free()
d_c.free()
del d_a, d_b, d_c
print('Device buffers released.')


Device buffers released.


## What changed from Chapter 2?

The CUDA kernel and input data are unchanged. Python now handles the host responsibilities through NumPy and PyCUDA: allocation, transfers, compilation, launch, validation, and display. The remaining examples in the sequence can build on this baseline to introduce other PyCUDA interfaces.

**Preparation status:** this notebook is delivered with empty outputs. Its structure, Python syntax, and the kernel's indexing and numerical logic were checked during preparation. CUDA compilation and execution require an NVIDIA GPU and were not performed in the preparation environment. Run all cells in a GPU runtime to obtain the actual results and timings.

Reference: [PyCUDA device interface and SourceModule documentation](https://documen.tician.de/pycuda/driver.html).
